**Looking for the recommended way to prepare ERA5 data?**

For most use cases you do not need to download raw ERA5 data manually. Use the
high-level helper `reskit.download_and_process`, which downloads exactly the
variables a given workflow needs, preprocesses them, and (optionally) tiles them
into the directory structure expected by `Era5Source`:

> ```python
> import reskit as rk
>
> result = rk.download_and_process(
>     workflows="openfield_pv_era5",
>     start_date="2000-01-01",
>     end_date="2000-12-31",
>     boundary_box={"north": 55, "south": 47, "west": 6, "east": 15},  # Germany
>     output_dir="/path/to/your/era5_data",
>     tiling=True,
> )
> ```

See [`1_1_3_prepare_era5_for_wind_workflow.ipynb`](1_1_3_prepare_era5_for_wind_workflow.ipynb)
and [`1_1_4_prepare_era5_for_solar_workflow.ipynb`](1_1_4_prepare_era5_for_solar_workflow.ipynb)
for full examples.

**The notebook below shows the lower-level, fully manual CDS download** — useful
when you need direct control over the variables, area, and timeframe being
requested.


# Download ERA5 Data

In order to download ERA5 weather data, you will need to set up an account at https://cds.climate.copernicus.eu/how-to-api. After registering, retrieve an API key and create a file called .cdsapirc in your home folder. It must contain

url: https://cds.climate.copernicus.eu/api.

key: PERSONAL-ACCESS-TOKEN.

Workflow:
1. Import required packages
2. Define directories to store downloaded data
3. Define variables and timeframes to download
4. Download data

In [ ]:
### imports

import cdsapi
import numpy as np
import os
import sys
from collections import OrderedDict
from os.path import join as jn
import pathlib

In [ ]:
### "raw_dir" this is the directory where the data will be downloaded, please set it as appropriate.
### "raw" must be contained in the name of the directory since we are downloading raw data
### Version control is also necessary.

version = "v2024-06-27"

# The following directory must exist (manually created by the user). /era5 and /raw subfolders will be created automatically if they do not exist.
# This is not automated to avoid accidentally creating files in the user OS.

raw_dir = pathlib.Path.cwd().joinpath(
    "era5", "raw"
)  # Change .cwd() to ("Path_to_your_directory") if you want to specify a different directory.
raw_dir.mkdir(exist_ok=True, parents=True)


version_dir = f"{raw_dir}/{version}"  # append with version label
print("Directory for specified version:", version_dir)

In [ ]:
### this are customisable input parameters:


### "variables": are a list of tuples (source, variable) according to the data you are interested in downloading
### "years_to_download": a list of years to be downloaded
### "months_to_download": a list of two-digit strings representing the month of the year
### "days_to_download": a list of two-digit strings representing the days of the month
### "hours_to_download": a list of strings representing the hours of the day
### "area_to_download": the geographical extent of the data that we want to download (Xmax,Ymin,Xmin,Ymax)


variables = [
    ("reanalysis-era5-single-levels", "100m_u_component_of_wind"),
    ("reanalysis-era5-single-levels", "100m_v_component_of_wind"),
    ("reanalysis-era5-single-levels", "surface_pressure"),
    ("reanalysis-era5-single-levels", "boundary_layer_height"),
    ("reanalysis-era5-single-levels", "2m_temperature"),
]

years_to_download = np.arange(2019, 2020, 1)

months_to_download = [
    "01",
    # "02",
    # "03",
    # "04",
    # "05",
    # "06",
    # "07",
    # "08",
    # "09",
    # "10",
    # "11",
    # "12",
]

days_to_download = [
    "01",
    "02",
    "03",
    "04",
    "05",
    "06",
    "07",
    "08",
    "09",
    "10",
    "11",
    "12",
    "13",
    "14",
    "15",
    "16",
    "17",
    "18",
    "19",
    "20",
    "21",
    "22",
    "23",
    "24",
    "25",
    "26",
    "27",
    "28",
    "29",
    "30",
    "31",
]

hours_to_download = [
    "00:00",
    "01:00",
    "02:00",
    "03:00",
    "04:00",
    "05:00",
    "06:00",
    "07:00",
    "08:00",
    "09:00",
    "10:00",
    "11:00",
    "12:00",
    "13:00",
    "14:00",
    "15:00",
    "16:00",
    "17:00",
    "18:00",
    "19:00",
    "20:00",
    "21:00",
    "22:00",
    "23:00",
]

area_to_download = [
    55,
    5,
    45,
    15,
]  # area for Germany, change if necessary

In [ ]:
#### this code does the following in a loop for the input parameters in the last cell:
### make sure that .cdsapi file is in your home directory
### 1) creates a "year_path" directory inside of the "raw_dir" directory if it is not already created
### 2) forms an 'OUTPUT' file path inside of the "year_path" directory and checks if it already exists
###     A) if it exists, it is assumed to be already downloaded and skipped
###     B) it not, it downloads it and calls it according to the "OUTPUT" file name

# Throw error if raw_dir does not exist
if not os.path.isdir(raw_dir):
    raise FileNotFoundError(f"The specified raw data directory {raw_dir} does not exist. Please create it manually.")

if not os.path.isdir(version_dir):
    os.mkdir(version_dir)
    print(version_dir, "was created.")

for year in years_to_download:
    year_path = jn(version_dir, str(year))
    if not os.path.isdir(year_path):
        os.mkdir(year_path)
        print(year_path, "was created.")

    print("DOWNLOADING YEAR:", year)

    for source, variable in variables:
        file_name = "{}.{}.{}.nc".format(source, year, variable)
        output_file = os.path.join(year_path, file_name)
        print(file_name)

        if os.path.isfile(output_file):
            print("already downloaded: skipped")

        else:
            c = cdsapi.Client()

            c.retrieve(
                source,
                {
                    "product_type": "reanalysis",
                    "area": area_to_download,
                    "variable": [
                        variable,
                    ],
                    "year": str(year),
                    "month": months_to_download,
                    "day": days_to_download,
                    "time": hours_to_download,
                    "format": "netcdf",
                },
                output_file,
            )

            print(" OPERATION DONE ")

    print("####################")